# 🎬 Media Downloader (Google Colab 雲端直存與遠端工作節點)

**雙模式支援：**
1. **🤖 模式一 (推薦)：Streamlit 本機聯動背景工作節點**：Colab 掛載後作為雲端 Worker 待命，您只需在 Mac 本地 Streamlit 介面貼網址按下載，Colab 自動在雲端抓取、封裝、直存 Google Drive，並將即時進度同步回您的 Mac 網頁！
2. **🖐️ 模式二：Colab 網頁獨立執行**：直接在下方表單填寫網址與儲存資料夾，點擊三角形按鈕直接下載。

## 步驟 1：安裝環境與掛載 Google Drive
執行下方儲存格安裝必要套件，並點擊授權掛載 Google Drive（兩模式皆需執行一次）。

In [ ]:
#@title ⚙️ 1. 環境安裝與掛載 Google Drive
# 安裝解析與高速下載依賴 (Colab 預設已內建 FFmpeg)
!pip install -q yt-dlp curl_cffi requests beautifulsoup4

from google.colab import drive
import os, sys

# 掛載 Google Drive
drive_mount_path = '/content/drive'
if not os.path.exists(drive_mount_path):
    drive.mount(drive_mount_path)
    print("✅ Google Drive 掛載成功！")
else:
    print("ℹ️ Google Drive 已在掛載狀態。")

# 載入 core_downloader 核心引擎
import urllib.request
raw_url = 'https://raw.githubusercontent.com/hangout1997/Media-Download/main/core_downloader.py'
print("正在由 GitHub 載入最新 core_downloader.py...")
urllib.request.urlretrieve(raw_url, '/content/core_downloader.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import core_downloader as cd
print("✅ 成功載入 core_downloader 核心引擎！")



---
## 🌟 模式一：啟動 Streamlit 聯動工作節點 (推薦)
執行下方儲存格後，Colab 會進入**待命監聽模式**。您只需回到 Mac 本機 Streamlit 網頁輸入網址，Colab 就會自動接單下載並回傳進度！

In [ ]:
#@title 🤖 啟動 Worker 待命監聽 (與 Mac 本地 Streamlit 聯動)
# 自動由 GitHub 下載最新版 colab_worker 引擎並執行 (保證永遠執行最新代碼)
!curl -s -L "https://raw.githubusercontent.com/hangout1997/Media-Download/main/colab_worker.py" -o /content/colab_worker.py
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import importlib
if 'colab_worker' in sys.modules:
    import colab_worker
    importlib.reload(colab_worker)
else:
    import colab_worker

colab_worker.run()



---
## 🖐️ 模式二：直接在 Colab 頁面貼網址下載 (獨立執行)
若您不想打開本地 Streamlit，也可直接在下方表單設定參數並點擊左側三角形執行。

In [ ]:
#@title 🚀 直接在 Colab 執行下載任務
#@markdown ### 網址與儲存設定：
video_urls = "https://www.mvffm.net/movies/your-name/" #@param {type:"string"}
save_folder = "Download" #@param {type:"string"}
only_audio = False #@param {type:"boolean"}

#@markdown ### 選填 Cookie (登入/風控解鎖)：
fb_cookie = "" #@param {type:"string"}
bili_cookie = "" #@param {type:"string"}

import os, time, sys

# 設定 Google Drive 目標儲存路徑
target_dir = os.path.join('/content/drive/MyDrive', save_folder.strip('/'))
os.makedirs(target_dir, exist_ok=True)

urls = [u.strip() for u in video_urls.strip().split('\n') if u.strip()]
cookies = {'fb_cookie': fb_cookie.strip(), 'bili_cookie': bili_cookie.strip()}

print(f"📁 Google Drive 目標資料夾: {target_dir}")
print(f"🔗 待處理網址數量: {len(urls)} 個\n" + "="*60)

for idx, raw_url in enumerate(urls, 1):
    url = cd.normalize_input_url(raw_url)
    print(f"\n📍 [{idx}/{len(urls)}] 正在解析: {url}")
    
    try:
        items = cd.get_media_items(url, cookies=cookies)
        if not items:
            print(f"❌ 找不到可用的串流或影片來源: {url}")
            continue
            
        for item in items:
            title = item.get('title', 'video')
            ext = 'mp3' if only_audio else item.get('ext', 'mp4')
            filename = f"{title}.{ext}"
            out_file = os.path.join(target_dir, filename)
            
            if os.path.exists(out_file):
                print(f"⏭️ 雲端已存在同名檔案，自動跳過: {filename}")
                continue
                
            print(f"📥 開始下載並直存: {filename}")
            
            last_pct = [-1]
            def _progress(pct, cur_dur, tot_dur, speed_mb, speed_x):
                cur_p = int(pct * 100)
                if cur_p != last_pct[0] and cur_p % 5 == 0:
                    last_pct[0] = cur_p
                    print(f"\r⏳ 進度: {cur_p}% | 速度: {speed_mb:.2f} MB/s ({speed_x:.2f}x)", end="", flush=True)
                    
            res_path = cd.download_media_to_path(
                item,
                out_path=out_file,
                force_audio=only_audio,
                cookies=cookies,
                progress_cb=_progress
            )
            print(f"\n🎉 成功直存至 Google Drive: {res_path} ({os.path.getsize(res_path)/1024/1024:.1f} MB)")
            
    except Exception as e:
        print(f"\n❌ 下載失敗 ({url}): {e}")

print("\n" + "="*60 + "\n🎊 所有下載與直存任務處理完畢！可直接至 Google Drive 查看檔案。")